In [1]:
import matplotlib as mpl
import numpy as np
from numpy import matrix
import matplotlib.pyplot as plt
from matplotlib.pyplot import figure
from matplotlib.pyplot import gcf
from matplotlib.lines import Line2D
import scipy as sp
from scipy import stats
from matplotlib import cm
import pandas as pd
import os
import math
#import seaborn as sns
from matplotlib.colors import LogNorm
import csv
import os.path as op
import numpy as np
import re
#import mesa_web
import xarray as xr

mpl.rcParams['mathtext.fontset'] = 'cm'
mpl.rc('font',family='Times New Roman')

# .nc to .csv

In [4]:
ds = xr.open_dataset('./data/WASP-39b/transit-spectrum-W39b-G395H-10pix_weighted-average.nc')
ds

<xarray.Dataset> Size: 11kB
Dimensions:              (central_wavelength: 344, bin_half_width: 344)
Coordinates:
  * central_wavelength   (central_wavelength) float64 3kB 2.763 2.77 ... 5.169
  * bin_half_width       (bin_half_width) float64 3kB 0.003438 ... 0.003377
Data variables:
    transit_depth        (central_wavelength) float64 3kB ...
    transit_depth_error  (central_wavelength) float64 3kB ...
Attributes:
    author:         H.R. Wakeford
    contact:        hannah.wakeford@bristol.ac.uk
    code:           paper_plots notebook under construction
    data_origin:    {"planet_spectra": "weighted mean of [DG,LF,MA,NE,NW,PAR,...
    doi:            none
    system_params:  {"rp": {"value": 1.0, "unit": "jupiterRad"}, "rs": {"valu...

In [5]:
transit_depth = ds['transit_depth']
transit_depth.to_pandas().to_csv('./data/WASP-39b/transit_depth.csv')
transit_depth_err = ds['transit_depth_error']
transit_depth_err.to_pandas().to_csv('./data/WASP-39b/transit_depth_error.csv')
transit_depth_err = ds['bin_half_width']
transit_depth_err.to_pandas().to_csv('./data/WASP-39b/bin_half_width.csv')

# Data Reduction Files to .dat

## Luis Files

In [6]:
path = "./data/GJ-3090b/Spectrum Files/"
new_path = "./data/GJ-3090b/"
filename = "GJ3090b_nirspec_g395h_nrs2_Stacked_exoTEDRF_R100"
new_filename = "GJ-3090b_NIRSpec_G395H_NRS2_Stacked_exoTEDRF_R100"

df = pd.read_csv(path+filename+".txt", sep = ' ')
wave = df["wvl(micron)"]
half_width = df["wvl_halfwidth(micron)"]
transit_depth = df["depths"]
transit_error = df["depths_err"]

df_t = pd.DataFrame()
df_t.insert(0,'0',wave)
df_t.insert(1,'1',half_width)
df_t.insert(2,'2',transit_depth)
df_t.insert(3,'3',transit_error)
df_t.to_csv(new_path + new_filename + ".dat", sep = " ", header=False, index=False)
df_t

,0,1,2,3
0,3.843277,0.019216,0.001430,0.000043
1,3.881903,0.019410,0.001405,0.000031
2,3.920917,0.019605,0.001406,0.000038
3,3.960323,0.019802,0.001452,0.000041
4,4.000125,0.020001,0.001512,0.000038
5,4.040328,0.020202,0.001478,0.000045
6,4.080934,0.020405,0.001517,0.000049
7,4.121948,0.020610,0.001463,0.000045
8,4.163375,0.020817,0.001449,0.000044
9,4.205218,0.021026,0.001476,0.000047


## Eureka! data

In [6]:
path = "./data/GJ-3090b/Spectrum Files/"
new_path = "./data/GJ-3090b/"
filename = "GJ3090_nirspec_g395h_nrs1"
new_filename = "GJ-3090b_NIRSpec_G395H_NRS2_Merged_Eureka_R250"

df = pd.read_csv(path+filename+".txt", sep = ' ')
wave = df["wvl(micron)"]
half_width = df["wvl_fullwidth(micron)"]*.5
transit_depth = df["depths"]
ted_list = df.loc[:,"depths_err_down"].tolist()
teu_list = df.loc[:,"depths_err_up"].tolist()
te_list = []
for i in range(0,len(ted_list)):
    te_list.append((ted_list[i]+teu_list[i])/2)
transit_error = pd.Series( (v for v in te_list) )

df_t = pd.DataFrame()
df_t.insert(0,'0',wave)
df_t.insert(1,'1',half_width)
df_t.insert(2,'2',transit_depth)
df_t.insert(3,'3',transit_error)
df_t.to_csv(new_path + new_filename + ".dat", sep = " ", header=False, index=False)
df_t

,0,1,2,3
0,2.712738,0.002713,0.001758,0.000443
1,2.723611,0.002724,0.001873,0.000283
2,2.734527,0.002735,0.001436,0.000206
3,2.745487,0.002745,0.001549,0.000154
4,2.756491,0.002756,0.001597,0.000128
...,...,...,...,...
150,5.085720,0.005086,0.001543,0.000086
151,5.106103,0.005106,0.001463,0.000072
152,5.126569,0.005127,0.001425,0.000079
153,5.147116,0.005147,0.001441,0.000083


## Eureka! data (Cyril)

In [ ]:
planet = "TOI-270c"
path = "./data/" + planet + "/Spectrum Files/"
new_path = "./data/" + planet + "/"
filename = "" + planet + "_nrs2_eureka"
new_filename = "" + planet + "_NIRSpec_G395H_NRS2_Eureka"

df = pd.read_csv(path+filename+".csv", sep = ', ')
wave = df["Central wavelength [microns]"]
half_width = df["Half-width of the wavelength bin [microns]"]
transit_depth = df["(R_p/R_s)^2 [ppm]"]*1e-6
ted_list = df.loc[:,"1 sigma lower error bar on (R_p/R_s)^2 [ppm]"].tolist()
teu_list = df.loc[:,"1 sigma upper error bar on (R_p/R_s)^2 [ppm]"].tolist()
te_list = []
for i in range(0,len(ted_list)):
    te_list.append(1e-6*(ted_list[i]+teu_list[i])/2)
transit_error = pd.Series( (v for v in te_list) )

df_t = pd.DataFrame()
df_t.insert(0,'0',wave)
df_t.insert(1,'1',half_width)
df_t.insert(2,'2',transit_depth)
df_t.insert(3,'3',transit_error)
df_t.to_csv(new_path + new_filename + ".dat", sep = " ", header=False, index=False)
df_t

/tmp/ipykernel_25520/4120271748.py:6: ParserWarning: Falling back to the 'python' engine because the 'c' engine does not support regex separators (separators > 1 char and different from '\s+' are interpreted as regex); you can avoid this warning by specifying engine='python'.
  df = pd.read_csv(path+filename+".csv", sep = ', ')


,0,1,2,3
0,3.8365,0.0135,0.003457,0.000049
1,3.8635,0.0135,0.003436,0.000050
2,3.8905,0.0135,0.003410,0.000050
3,3.9175,0.0135,0.003461,0.000052
4,3.9445,0.0135,0.003521,0.000053
5,3.9715,0.0135,0.003429,0.000054
6,3.9985,0.0135,0.003501,0.000055
7,4.0255,0.0135,0.003495,0.000055
8,4.0525,0.0135,0.003559,0.000056
9,4.0795,0.0135,0.003425,0.000056


## exoTEDRF data

In [ ]:
path = "./data/GJ-3090b/Spectrum Files/"
filename = "GJ-3090b_NIRISS_SOSS_Ord1_Visit1"

df = pd.read_csv(path+filename+".txt", sep = ',')
wave = df["wave"]
half_width = df["wave_err"]
transit_depth = df["dppm"]*1e-6
transit_error = df["dppm_err"]*1e-6

df_t = pd.DataFrame()
df_t.insert(0,'0',wave)
df_t.insert(1,'1',half_width)
df_t.insert(2,'2',transit_depth)
df_t.insert(3,'3',transit_error)
df_t.to_csv(path + filename + ".dat", sep = " ", header=False, index=False)
df_t

## NAMELESS data

In [16]:
path = "./data/GJ-3090b/Text Files/"
new_path = "./data/GJ-3090b/"
filename = "GJ3090_NIRISS_ord2_visit2_NAMELESS_R50"
new_filename = "GJ-3090b_NIRISS_SOSS_Ord2_Visit2_NAMELESS_R50"

df = pd.read_csv(path+filename+".txt", sep = ' ')
wave = df["Wavelengthbincentre"]
half_width = df["wavelengthbinfullwidth"]*.5
transit_depth = df["Transitdepth"]
ted_list = df.loc[:,"Transitdepth-veerror"].tolist()
teu_list = df.loc[:,"Transitdepth+veerror"].tolist()
te_list = []
for i in range(0,len(ted_list)):
    te_list.append((ted_list[i]+teu_list[i])/2)
transit_error = pd.Series( (v for v in te_list) )

df_t = pd.DataFrame()
df_t.insert(0,'0',wave)
df_t.insert(1,'1',half_width)
df_t.insert(2,'2',transit_depth)
df_t.insert(3,'3',transit_error)
df_t.to_csv(new_path + new_filename + ".dat", sep = " ", header=False, index=False)
df_t

,0,1,2,3
0,0.606858,0.006069,0.001537,0.000116
1,0.619117,0.006191,0.001552,0.000115
2,0.631625,0.006316,0.001536,0.000101
3,0.644385,0.006444,0.001653,0.000094
4,0.657403,0.006574,0.001615,0.000083
5,0.670684,0.006707,0.001693,0.000083
6,0.684233,0.006842,0.001598,0.000081
7,0.698056,0.006980,0.001546,0.000068
8,0.712158,0.007122,0.001689,0.000069
9,0.726545,0.007266,0.001603,0.000063


# ExoTEP

In [4]:
planet = "TOI-270c"
planet = "TOI-1685b"
path = "./data/" + planet + "/Spec files/"
new_path = "./data/" + planet + "/"
filename = "NIRISS_SOSS_EXCLUDED"
new_filename = "" + planet + "_NIRISS_SOSS_exc_ExoTEDRF"

df = pd.read_csv(path+filename+".txt", sep = ',')
wave = df["wave"]
wavemin_list = df.loc[:,"waveMin"].tolist()
wavemax_list = df.loc[:,"waveMax"].tolist()
halfwidth_list = []
for i in range(0,len(wavemin_list)):
    halfwidth_list.append((wavemax_list[i]-wavemin_list[i])/2)
halfwidth = pd.Series( (v for v in halfwidth_list) )
transit_depth = df["yval"]*10**(-6)
ted_list = df.loc[:,"yerrLow"].tolist()
teu_list = df.loc[:,"yerrUpp"].tolist()
te_list = []
for i in range(0,len(ted_list)):
    te_list.append(1e-6*(ted_list[i]+teu_list[i])/2)
transit_error = pd.Series( (v for v in te_list) )

df_t = pd.DataFrame()
df_t.insert(0,'0',wave)
df_t.insert(1,'1',halfwidth)
df_t.insert(2,'2',transit_depth)
df_t.insert(3,'3',transit_error)
df_t.to_csv(new_path + new_filename + ".dat", sep = " ", header=False, index=False)#, float_format='%.9f')
print(df_t)

            0         1         2         3
0    0.604064  0.003080  0.000409  0.000346
1    0.610031  0.002887  0.001354  0.000568
2    0.616027  0.003109  0.000898  0.000494
3    0.622260  0.003123  0.000841  0.000532
4    0.628522  0.003138  0.000614  0.000353
..        ...       ...       ...       ...
150  2.710268  0.013504  0.000851  0.000143
151  2.737306  0.013534  0.000599  0.000128
152  2.764657  0.013816  0.000548  0.000139
153  2.792318  0.013845  0.000747  0.000142
154  2.812979  0.006815  0.000979  0.000224

[155 rows x 4 columns]


## ExoTEP (Wavelength bins not defined)

#### If there are large gaps between instrument modes (e.g., NRS1 and NRS2, this estimation will not work well and will need to be fixed at these endpoints)

In [ ]:
path = "./data/TOI-1685b/Spec files/"
new_path = "./data/TOI-1685b/"
filename = "NIRSPEC_G395H_ONLY"
new_filename = "TOI-1685b_NIRSpec_G395H_Stacked_ExoTEP_R100"

df = pd.read_csv(path+filename+".txt", sep = ',')
wave = df["wave"]
# wavemin_list = df.loc[:,"waveMin"].tolist()
# wavemax_list = df.loc[:,"waveMax"].tolist()
halfwidth_list = []
for i in range(0,len(wave)-1):
    halfwidth_list.append((wave[i+1]-wave[i])/2)
halfwidth_list.append((wave[i+1]-wave[i])/2)
halfwidth = pd.Series( (v for v in halfwidth_list) )
transit_depth = df["yval"]*10**(-6)
ted_list = df.loc[:,"yerrLow"].tolist()
teu_list = df.loc[:,"yerrUpp"].tolist()
te_list = []
for i in range(0,len(ted_list)):
    te_list.append(1e-6*(ted_list[i]+teu_list[i])/2)
transit_error = pd.Series( (v for v in te_list) )

df_t = pd.DataFrame()
df_t.insert(0,'0',wave)
df_t.insert(1,'1',halfwidth)
df_t.insert(2,'2',transit_depth)
df_t.insert(3,'3',transit_error)
df_t.to_csv(new_path + new_filename + ".dat", sep = " ", header=False, index=False, float_format='%.9f')
df_t

,0,1,2,3
0,0.604240,0.002981,0.000485,0.000338
1,0.610201,0.002995,0.000921,0.000465
2,0.616191,0.003113,0.000858,0.000424
3,0.622417,0.003128,0.000510,0.000347
4,0.628673,0.003143,0.000682,0.000372
...,...,...,...,...
151,2.712173,0.013520,0.000924,0.000129
152,2.739213,0.013549,0.000650,0.000141
153,2.766311,0.013832,0.000596,0.000134
154,2.793975,0.009826,0.000838,0.000152
